# 📘 Aula 10g: Transfer Learning em NLP – Embeddings Pré-treinados com GloVe (Stanford) & Bidirectional LSTM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula10g%20-%20glove.ipynb)

Na aula anterior (Aula 10f), treinamos nossa camada de Word Embedding do zero a partir exclusivamente das 20.000 resenhas de treino. Embora tenhamos obtido bons resultados, aprender a semântica da linguagem humana exige bilhões de palavras de contexto.

Nesta aula, realizamos a revolução do **Transfer Learning aplicado ao Processamento de Linguagem Natural**: utilizaremos os **Word Embeddings pré-treinados GloVe (Global Vectors for Word Representation)**, desenvolvidos por pesquisadores da Universidade de Stanford. O GloVe foi treinado sobre um corpus gigantesco de **6 bilhões de palavras** (Wikipedia 2014 + Gigaword 5), capturando relações analógicas espantosas (como $\vec{v}_{\text{rei}} - \vec{v}_{\text{homem}} + \vec{v}_{\text{mulher}} \approx \vec{v}_{\text{rainha}}$). Aprenderemos a carregar esses vetores, construir uma matriz de pesos congelada e integrá-la a uma rede **Bidirectional LSTM**!

---

### 🎯 Objetivo da Aula
1. Baixar e descompactar os vetores oficiais do GloVe 6B da Universidade de Stanford.
2. Fazer o parsing do arquivo de texto bruto `glove.6B.100d.txt` mapeando 400.000 palavras para vetores NumPy de 100 dimensões.
3. Construir a **Matriz de Embedding (`embedding_matrix`)** alinhando o vocabulário do Keras com os vetores pré-treinados do GloVe.
4. Instanciar a camada `Embedding` com pesos congelados (`trainable=False`), impedindo que o treinamento perturbe o conhecimento semântico pré-existente.
5. Conectar a camada congelada a uma rede **Bidirectional LSTM** com `mask_zero=True` e `Dropout(0.5)`.
6. Avaliar o modelo no teste independente do IMDB e consolidar a comparação entre Embeddings do zero versus pré-treinados.

---

### 💡 A Intenção Pedagógica do Professor
* **Transfer Learning Semelhante à Visão Computacional**: Assim como congelamos a base convolucional da VGG16 (treinada na ImageNet) na Aula 08a, aqui congelamos os vetores semânticos do GloVe (treinados na Wikipedia), aplicando o mesmo princípio unificador do Deep Learning!
* **Parsing Manual de Arquivos de Embeddings**: Ensinar o algoritmo exato de leitura linha a linha de formatos `.txt` de embeddings (usado por GloVe, Word2Vec e FastText).

---

### 📚 Glossário da Aula
* **GloVe (Global Vectors)**: Algoritmo de aprendizado não supervisionado que obtém representações vetoriais de palavras mapeando coocorrências globais em grandes corpora.
* **Embedding Matrix**: Matriz $N \times D$ onde a linha $i$ contém o vetor contínuo de $D$ dimensões correspondente à palavra de índice $i$ no vocabulário.
* **Frozen Embeddings (Pesos Congelados)**: Fixação dos pesos da camada `Embedding` (`trainable = False`), reduzindo o tempo de convergência e evitando overfitting em datasets pequenos.

---

### 🧭 Trilha da Aula
1. [Download e Preparação do Dataset IMDB](#1-download-e-extração-do-dataset-imdb)
2. [Vetorização Sequencial com TextVectorization](#2-vetorização-sequencial-no-keras)
3. [Download dos Vetores Pré-Treinados GloVe 6B](#3-download-do-glove-6b-stanford)
4. [Parsing e Carregamento do Arquivo glove.6B.100d.txt](#4-parsing-do-arquivo-de-vetores-glove)
5. [Construção da Matriz de Embeddings Alinhada](#5-alinhamento-da-matriz-de-embeddings)
6. [Camada Embedding Congelada (trainable=False)](#6-criação-da-camada-embedding-congelada)
7. [Arquitetura Neural com Bidirectional LSTM](#7-arquitetura-com-bilstm-e-dropout)
8. [Treinamento, Checkpoint e Avaliação no Teste](#8-treinamento-e-avaliação-final)


## 1. As Perguntas Centrais de NLP

* *"Qual é o tópico deste texto?"* (classificação)
* *"Este texto contém ofensas?"* (moderação)
* *"Este comentário expressa sentimento positivo ou negativo?"* **(Análise de Sentimentos)**


## 2. Download e Extração do Dataset IMDB

**O que estamos fazendo**: Baixar e descompactar o arquivo oficial do corpus de filmes do Stanford AI Lab.

**Por que fazer assim**: Fornece os 50.000 textos de resenhas para treino e teste.



In [ ]:
# Download e descompactação do dataset IMDB
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz


## 3. Limpeza do Diretório Não Supervisionado

**O que estamos fazendo**: Excluir os dados não rotulados da pasta `unsup`.

**Por que fazer assim**: Mantém apenas as classes binárias supervisionadas `pos` e `neg`.



In [ ]:
# Remove dados sem anotação
!rm -r aclImdb/train/unsup


## 4. Inspeção de uma Resenha Positiva Real

**O que estamos fazendo**: Exibir o texto bruto do arquivo `4077_10.txt`.

**Por que fazer assim**: Mostra a riqueza vocabular e as nuances estilísticas presentes nas opiniões dos espectadores.



In [ ]:
# Exibe uma resenha de exemplo
!cat aclImdb/train/pos/4077_10.txt


## 5. Criação da Partição de Validação com pathlib

**O que estamos fazendo**: Mover aleatoriamente 20% das resenhas de treino para `val` com semente fixa (1337).

**Por que fazer assim**: Garante uma partição independente de 5.000 resenhas para validação de época e salvamento de checkpoints.



In [ ]:
import os, pathlib, shutil, random

base_dir = pathlib.Path('aclImdb')
val_dir = base_dir / 'val'
train_dir = base_dir / 'train'

for category in ('neg', 'pos'):
  os.makedirs(val_dir / category, exist_ok=True)
  files = os.listdir(train_dir / category)
  random.Random(1337).shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname,
                val_dir / category / fname)
print('Diretório de validação configurado!')


## 6. Streaming com text_dataset_from_directory

**O que estamos fazendo**: Instanciar geradores de lotes de 32 textos para `train`, `val` e `test`.

**Por que fazer assim**: Lê os textos sob demanda a partir do disco.



In [ ]:
from tensorflow import keras
batch_size = 32

print('Carregando treino...')
train_ds = keras.utils.text_dataset_from_directory('aclImdb/train', batch_size=batch_size)
print('Carregando validação...')
val_ds = keras.utils.text_dataset_from_directory('aclImdb/val', batch_size=batch_size)
print('Carregando teste...')
test_ds = keras.utils.text_dataset_from_directory('aclImdb/test', batch_size=batch_size)


## 7. Inspeção dos Lotes de Texto

**O que estamos fazendo**: Verificar formato dos tensores de strings e rótulos de classe.

**Por que fazer assim**: Confirma lotes de 32 resenhas de texto livre com rótulos binários 0 ou 1.



In [ ]:
for inputs, targets in train_ds:
  print('Formato do lote de textos:  ', inputs.shape)
  print('Formato do lote de rótulos: ', targets.shape)
  print('Primeiro texto bruto:\n', inputs[0].numpy().decode('utf-8')[:250], '...')
  print('Rótulo supervisionado:      ', targets[0].numpy())
  break


## 8. Vetorização Sequencial Inteira (TextVectorization)

**O que estamos fazendo**: Configurar `output_mode='int'` e `output_sequence_length=600`, limitando ao vocabulário de 20.000 tokens.

**Por que fazer assim**: Gera tensores inteiros de dimensão `(32, 600)` preservando a ordem sintática das palavras.

> [!NOTE]
> Tokens excedentes são truncados em 600 palavras e textos menores são preenchidos com zeros (*padding*).



In [ ]:
from tensorflow.keras.layers import TextVectorization

max_length = 600
max_tokens = 20000

text_vectorization = TextVectorization(max_tokens=max_tokens,
                                       output_mode='int',
                                       output_sequence_length=max_length)

# Ajusta o vocabulário nos textos de treino
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

# Aplica a vetorização sequencial nas partições com paralelismo
binary_train_ds = train_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_val_ds = val_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
binary_test_ds = test_ds.map(lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)


## 9. Inspeção dos Tensores Finais

**O que estamos fazendo**: Confirmar formato e tipos de dados dos lotes inteiros.

**Por que fazer assim**: Cada resenha vira um vetor de 600 inteiros.



In [ ]:
for inputs, targets in binary_train_ds:
  print('Formato das sequências inteiras:', inputs.shape)
  print('Primeiros 30 tokens inteiros da amostra 0:\n', inputs[0].numpy()[:30])
  break


## 10. Download dos Vetores Pré-Treinados GloVe 6B (Stanford)

**O que estamos fazendo**: Baixar o pacote `glove.6B.zip` dos servidores da Universidade de Stanford e descompactar.

**Por que fazer assim**: Contém arquivos de embeddings com diferentes dimensões (50d, 100d, 200d, 300d) calculados sobre 6 bilhões de palavras da Wikipedia e Gigaword.

> [!TIP]
> Utilizaremos a versão de 100 dimensões (`glove.6B.100d.txt`), que oferece o equilíbrio ideal entre riqueza semântica e velocidade de treino!



In [ ]:
# Baixa os vetores GloVe de Stanford e descompacta
!wget http://nlp.stanford.edu/data/glove.6B.zip --no-check-certificate
!unzip -q glove.6B.zip


## 11. Parsing do Arquivo glove.6B.100d.txt

**O que estamos fazendo**: Ler o arquivo de texto linha por linha e montar o dicionário `embeddings_index` mapeando cada palavra para seu vetor de coeficientes numéricos.

**Por que fazer assim**: O arquivo contém 400.000 linhas, onde cada linha possui a palavra seguida de 100 números float representando suas coordenadas no espaço semântico.



In [ ]:
import numpy as np

path_to_glove_file = 'glove.6B.100d.txt'

# Dicionário para armazenar o mapeamento: palavra -> vetor NumPy de 100 dimensões
embeddings_index = {}
with open(path_to_glove_file, encoding='utf-8') as f:
  for line in f:
    word, coefs = line.split(maxsplit=1)
    coefs = np.fromstring(coefs, 'f', sep=' ')
    embeddings_index[word] = coefs

print(f'Total de palavras carregadas do GloVe: {len(embeddings_index)}')


## 12. Alinhamento da Matriz de Embeddings e Camada Congelada

**O que estamos fazendo**: Construir a `embedding_matrix` alinhando os índices do vocabulário do Keras aos vetores do GloVe, e instanciar a camada `Embedding` com pesos constantes e `trainable=False`.

**Por que fazer assim**: Como os índices de palavras do Keras diferem da ordem do arquivo do GloVe, precisamos de uma matriz alinhada onde a linha $i$ corresponda exatamente à palavra de índice $i$.

> [!NOTE]
> `trainable=False` congela os pesos da matriz, preservando os vetores semânticos aprendidos na Wikipedia intactos durante todo o treinamento!

> [!TIP]
> `mask_zero=True` garante que a LSTM ignore automaticamente os zeros de padding!



In [ ]:
from tensorflow.keras.layers import Embedding

# Dimensões da matriz de embeddings
num_tokens = len(text_vectorization.get_vocabulary()) + 2
embedding_dim = 100
vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(range(len(vocabulary)), vocabulary))

# Inicializa matriz de zeros
embedding_matrix = np.zeros((num_tokens, embedding_dim))
for word, i in word_index.items():
  embedding_vector = embeddings_index.get(word)
  if embedding_vector is not None:
    embedding_matrix[i] = embedding_vector

# Cria a camada de Embedding pré-inicializada e congelada
embedding_layer = Embedding(
    num_tokens,
    embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,  # Congela os pesos (Transfer Learning em NLP)
    mask_zero=True    # Mascara zeros de padding
)


## 13. Arquitetura Neural com GloVe e Bidirectional LSTM

**O que estamos fazendo**: Montar o modelo conectando: Entrada -> `embedding_layer` (congelada) -> `Bidirectional(LSTM(32))` -> `Dropout(0.5)` -> `Dense(1, sigmoid)`.

**Por que fazer assim**: A rede utiliza o conhecimento semântico da Wikipedia e foca todo o treinamento exclusivamente em aprender a classificar sentimentos com base nesses vetores ricos.

> [!NOTE]
> Observe no sumário do modelo a grande quantidade de parâmetros não treináveis (*Non-trainable params*), que correspondem aos pesos congelados do GloVe!



In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=20000, hidden_dim=32):
  """
  Rede neural para análise de sentimentos com embeddings GloVe congelados e BiLSTM.
  """
  inputs = keras.Input(shape=(None,), dtype='int64')
  # Aplica a camada de embedding pré-treinada do GloVe
  embedded = embedding_layer(inputs)
  # LSTM Bidirecional processando os vetores semânticos
  x = layers.Bidirectional(layers.LSTM(hidden_dim))(embedded)
  x = layers.Dropout(0.5)(x)
  outputs = layers.Dense(1, activation='sigmoid')(x)
  
  model = keras.Model(inputs, outputs)
  model.compile(optimizer='rmsprop',
                loss='binary_crossentropy',
                metrics=['accuracy'])
  return model

model = get_model()
model.summary()


## 14. Treinamento e Avaliação no Teste

**O que estamos fazendo**: Treinar o modelo por 10 épocas salvando o checkpoint `glove.keras` e avaliar nos 25.000 textos de teste do IMDB.

**Por que fazer assim**: Mede o impacto de utilizar representações semânticas pré-treinadas para tarefas de NLP com dados de treino limitados.



In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint('glove.keras', save_best_only=True)
]

history = model.fit(binary_train_ds,
                    validation_data=binary_val_ds,
                    epochs=10,
                    callbacks=callbacks)

# Carrega o melhor checkpoint e avalia no teste
model = keras.models.load_model('glove.keras')
test_loss, test_acc = model.evaluate(binary_test_ds)
print(f'Perda no Teste (GloVe + BiLSTM):    {test_loss:.3f}')
print(f'Acurácia no Teste (GloVe + BiLSTM): {test_acc * 100:.2f}%')


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, realizamos a aplicação completa de **Transfer Learning em Processamento de Linguagem Natural**:

1. **Stanford GloVe**: Compreendemos a formulação do GloVe e sua capacidade de capturar geometrias semânticas complexas a partir de bilhões de palavras.
2. **Parsing e Indexação**: Lemos e indexamos 400.000 vetores de 100 dimensões diretamente a partir do arquivo bruto `.txt`.
3. **Alinhamento Matricial**: Construímos a `embedding_matrix` alinhando a ordem interna do vocabulário do Keras com os vetores do GloVe.
4. **Pesos Congelados**: Instanciamos `Embedding(trainable=False)`, economizando processamento de GPU e protegendo o conhecimento pré-treinado contra sobreajuste.
5. **Conexão Recorrente**: Aliamos os vetores semânticos à capacidade sequencial da **Bidirectional LSTM**, concluindo o ciclo do NLP recorrente moderno.

---

## 👉 Próximo Passo
No **Lote 11 (Aulas 11a a 11d)**, entraremos na era mais revolucionária da história da Inteligência Artificial: os **Mecanismos de Auto-Atenção (Self-Attention)**, a base matemática de todas as Large Language Models (LLMs) modernas!
